In [10]:
from typing import List

from agent.components.GaussianProcess import GASK
from notebooks.summersoc.globals import PATH_METRICS_DEMO_EXPLORE, PATH_MODEL_LIST, SPLIT_DATA_INTO_X_PARTS, \
    ITERATE_THROUGH_X_PARTS, DIR_CANDIDATES
%load_ext autoreload
%autoreload 2

import matplotlib.pyplot as plt
import pandas as pd
import joblib

plt.style.use('default')

from agent.components import RASK
from agent.components.commons import ServiceType, theoretical_param_bounds, ServiceVar
from agent.components.commons import SloSet

services = [ServiceType.QR, ServiceType.CV, ServiceType.PC]
slos = [SloSet.DEFAULT, SloSet.HIGH_PERF, SloSet.LOW_COST, SloSet.HIGH_QUALITY]

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [8]:
df_explore = pd.read_csv(PATH_METRICS_DEMO_EXPLORE)
df_explore_preprocessed = RASK.preprocess_data(df_explore)

gp_list = joblib.load(PATH_MODEL_LIST)['gp_list']
rm_list = joblib.load(PATH_MODEL_LIST)['rm_list']

INFO:multiscale:Training data contains service types <StringArray>
[  'elastic-workbench-qr-detector',   'elastic-workbench-cv-analyzer',
 'elastic-workbench-pc-visualizer']
Length: 3, dtype: str


## **Plan**: Intent-based Inference

Find optimal parameter configs for each SLO x Service combination after seeing certain data shares

**TODO**: This here could actually be the place where we introduce out multi-dimensional methodology

![Monitoring Infrastructure](img/RASK_Methodology.jpg)

### Infer parameter assignments

#### Variant 1: Using deterministic gradient descent

In [3]:
from agent.components.Optimizer import run_optimizer_multi


def get_all_inferred_assignments(model_list: List[GASK] | List[RASK.RASK]):
    all_inferred_assignments = []

    for i in range(ITERATE_THROUGH_X_PARTS):
        for q in slos:
            for s in services:
                data_ratio = (i + 1) / ITERATE_THROUGH_X_PARTS
                if type(model_list[0]) is RASK.RASK:
                    model: RASK.RASK = model_list[i]
                else:
                    model: GASK = model_list[i][s]

                # Run optimizer to find the best configuration
                solutions = run_optimizer_multi(s, q.value, model, theoretical_param_bounds[s], runs=10)
                fitness, config = max(solutions, key=lambda x: x[0])

                # Predict performance (mu, sigma) for the chosen configuration
                x_state = {ServiceVar.COST: config[0], ServiceVar.QUALITY: config[1]}
                x_state = x_state | ({ServiceVar.MODEL: config[2]} if s == ServiceType.CV else {})
                pred = model.predict(s, "max_tp", x_state)  # this gives a distribution for GP

                # Store everything needed for the next block
                # We include empirical_var_bounds here as it changes per iteration
                all_inferred_assignments.append({
                    'data_rate': data_ratio,
                    'service_type': s.value,
                    'slo_set': q.name,
                    'p_fitness': fitness,
                    'dist': pred,
                    'cores': x_state[ServiceVar.COST],
                    'data_quality': x_state[ServiceVar.QUALITY],
                    'model_size': x_state[ServiceVar.MODEL] if s == ServiceType.CV else -1,
                })

                print(f"Predicted fitness for {s.value} and {q.name} with {data_ratio * 100}% training data: {fitness}")
    return all_inferred_assignments


In [4]:
# rm_assignments = get_all_inferred_assignments(rm_list)

#### Variant 2: Using stochastic gradient descent

In [5]:
gp_assignments = get_all_inferred_assignments(gp_list)

Predicted fitness for elastic-workbench-qr-detector and DEFAULT with 1.6666666666666667% training data: 0.89579311653406
Predicted fitness for elastic-workbench-cv-analyzer and DEFAULT with 1.6666666666666667% training data: 0.027790157045518633
Predicted fitness for elastic-workbench-pc-visualizer and DEFAULT with 1.6666666666666667% training data: 0.8957995250660532
Predicted fitness for elastic-workbench-qr-detector and HIGH_PERF with 1.6666666666666667% training data: 0.9841860643816022
Predicted fitness for elastic-workbench-cv-analyzer and HIGH_PERF with 1.6666666666666667% training data: 0.9469885937568572
Predicted fitness for elastic-workbench-pc-visualizer and HIGH_PERF with 1.6666666666666667% training data: 0.9842030373175783
Predicted fitness for elastic-workbench-qr-detector and LOW_COST with 1.6666666666666667% training data: 0.7158252736746075
Predicted fitness for elastic-workbench-cv-analyzer and LOW_COST with 1.6666666666666667% training data: 0.7082645714648029
Pred

Predicted fitness for elastic-workbench-cv-analyzer and HIGH_QUALITY with 15.0% training data: 0.95
Predicted fitness for elastic-workbench-pc-visualizer and HIGH_QUALITY with 15.0% training data: 0.8495674861254072
Predicted fitness for elastic-workbench-qr-detector and DEFAULT with 16.666666666666664% training data: 0.35880160194896543
Predicted fitness for elastic-workbench-cv-analyzer and DEFAULT with 16.666666666666664% training data: 0.67
Predicted fitness for elastic-workbench-pc-visualizer and DEFAULT with 16.666666666666664% training data: 0.8203156208315278
Predicted fitness for elastic-workbench-qr-detector and HIGH_PERF with 16.666666666666664% training data: 0.8380687416706404
Predicted fitness for elastic-workbench-cv-analyzer and HIGH_PERF with 16.666666666666664% training data: 0.9500000000000001
Predicted fitness for elastic-workbench-pc-visualizer and HIGH_PERF with 16.666666666666664% training data: 0.8934754359233142
Predicted fitness for elastic-workbench-qr-detect

Predicted fitness for elastic-workbench-qr-detector and HIGH_QUALITY with 18.333333333333332% training data: 0.6395440735321786
Predicted fitness for elastic-workbench-cv-analyzer and HIGH_QUALITY with 18.333333333333332% training data: 0.95
Predicted fitness for elastic-workbench-pc-visualizer and HIGH_QUALITY with 18.333333333333332% training data: 0.8564280544305289
Predicted fitness for elastic-workbench-qr-detector and DEFAULT with 20.0% training data: 0.3539687726657486
Predicted fitness for elastic-workbench-cv-analyzer and DEFAULT with 20.0% training data: 0.67
Predicted fitness for elastic-workbench-pc-visualizer and DEFAULT with 20.0% training data: 0.8216013800865437
Predicted fitness for elastic-workbench-qr-detector and HIGH_PERF with 20.0% training data: 0.8357441004865083
Predicted fitness for elastic-workbench-cv-analyzer and HIGH_PERF with 20.0% training data: 0.9500000000000001
Predicted fitness for elastic-workbench-pc-visualizer and HIGH_PERF with 20.0% training dat

Predicted fitness for elastic-workbench-qr-detector and DEFAULT with 25.0% training data: 0.43898627459667733
Predicted fitness for elastic-workbench-cv-analyzer and DEFAULT with 25.0% training data: 0.67
Predicted fitness for elastic-workbench-pc-visualizer and DEFAULT with 25.0% training data: 0.7021138742034
Predicted fitness for elastic-workbench-qr-detector and HIGH_PERF with 25.0% training data: 0.8484085210183715
Predicted fitness for elastic-workbench-cv-analyzer and HIGH_PERF with 25.0% training data: 0.9500000000000001
Predicted fitness for elastic-workbench-pc-visualizer and HIGH_PERF with 25.0% training data: 0.9078137247780789
Predicted fitness for elastic-workbench-qr-detector and LOW_COST with 25.0% training data: 0.8165629418476064
Predicted fitness for elastic-workbench-cv-analyzer and LOW_COST with 25.0% training data: 0.8015462710811132
Predicted fitness for elastic-workbench-pc-visualizer and LOW_COST with 25.0% training data: 0.8930259200560978
Predicted fitness fo

Predicted fitness for elastic-workbench-pc-visualizer and HIGH_QUALITY with 25.0% training data: 0.7577917940899594
Predicted fitness for elastic-workbench-qr-detector and DEFAULT with 26.666666666666668% training data: 0.43566494553793583
Predicted fitness for elastic-workbench-cv-analyzer and DEFAULT with 26.666666666666668% training data: 0.67
Predicted fitness for elastic-workbench-pc-visualizer and DEFAULT with 26.666666666666668% training data: 0.7032681427939872
Predicted fitness for elastic-workbench-qr-detector and HIGH_PERF with 26.666666666666668% training data: 0.8470712169937604
Predicted fitness for elastic-workbench-cv-analyzer and HIGH_PERF with 26.666666666666668% training data: 0.9500000000000001
Predicted fitness for elastic-workbench-pc-visualizer and HIGH_PERF with 26.666666666666668% training data: 0.9109882637549399
Predicted fitness for elastic-workbench-qr-detector and LOW_COST with 26.666666666666668% training data: 0.8276375387286681
Predicted fitness for ela

Predicted fitness for elastic-workbench-pc-visualizer and HIGH_QUALITY with 33.33333333333333% training data: 0.7605659556279566
Predicted fitness for elastic-workbench-qr-detector and DEFAULT with 35.0% training data: 0.41106130614965847
Predicted fitness for elastic-workbench-cv-analyzer and DEFAULT with 35.0% training data: 0.67
Predicted fitness for elastic-workbench-pc-visualizer and DEFAULT with 35.0% training data: 0.7050663306894437
Predicted fitness for elastic-workbench-qr-detector and HIGH_PERF with 35.0% training data: 0.8137510449452457
Predicted fitness for elastic-workbench-cv-analyzer and HIGH_PERF with 35.0% training data: 0.9500000000000001
Predicted fitness for elastic-workbench-pc-visualizer and HIGH_PERF with 35.0% training data: 0.9174129329121804
Predicted fitness for elastic-workbench-qr-detector and LOW_COST with 35.0% training data: 0.8491320266937545
Predicted fitness for elastic-workbench-cv-analyzer and LOW_COST with 35.0% training data: 0.8672018760455374


Predicted fitness for elastic-workbench-qr-detector and HIGH_QUALITY with 38.333333333333336% training data: 0.40323214128488766
Predicted fitness for elastic-workbench-cv-analyzer and HIGH_QUALITY with 38.333333333333336% training data: 0.95
Predicted fitness for elastic-workbench-pc-visualizer and HIGH_QUALITY with 38.333333333333336% training data: 0.7603933339922322
Predicted fitness for elastic-workbench-qr-detector and DEFAULT with 40.0% training data: 0.4279949930428935
Predicted fitness for elastic-workbench-cv-analyzer and DEFAULT with 40.0% training data: 0.6696605372679649
Predicted fitness for elastic-workbench-pc-visualizer and DEFAULT with 40.0% training data: 0.7047166592315071
Predicted fitness for elastic-workbench-qr-detector and HIGH_PERF with 40.0% training data: 0.8431807659637844
Predicted fitness for elastic-workbench-cv-analyzer and HIGH_PERF with 40.0% training data: 0.9499153369395287
Predicted fitness for elastic-workbench-pc-visualizer and HIGH_PERF with 40.

#### Export to candidate solution script, with each config repeated x times

In [11]:

candidate_repetitions = []
runs_per_config = 50

# Iterate through the list in steps of 3 (the size of your service triple)
for i in range(0, len(gp_assignments), 3):
    # Extract the current triple of rows
    triple = gp_assignments[i: i + 3]

    # Repeat this specific triple for the number of runs
    for run_idx in range(runs_per_config):
        for row in triple:
            new_row = row.copy()
            new_row['rep'] = run_idx + 1
            candidate_repetitions.append(new_row)

file_name = f'candidate_solutions_{ITERATE_THROUGH_X_PARTS}_{SPLIT_DATA_INTO_X_PARTS}_{runs_per_config}.csv'
df_candidates = pd.DataFrame(candidate_repetitions)
df_candidates.to_csv(DIR_CANDIDATES / file_name, index=False)
